# hERG Inhibition Prediction - Exploration Notebook

This notebook walks through the full pipeline for hERG prediction:
1. Data download and preprocessing
2. Exploratory data analysis
3. Feature engineering
4. Model training and comparison
5. Evaluation and analysis

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import Draw

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Download Data from ChEMBL

In [ ]:
from herg_predictor.data import download_herg_data

# Download raw data (this may take a few minutes)
df_raw = download_herg_data(output_path='../data/raw/herg_chembl.csv')
print(f"Downloaded {len(df_raw)} records")

## 2. Preprocess Data

In [ ]:
from herg_predictor.data import preprocess_herg_data

df = preprocess_herg_data(
    input_path='../data/raw/herg_chembl.csv',
    output_path='../data/processed/herg_clean.parquet',
    activity_threshold_nm=10000,  # 10 µM
)
df.head()

## 3. Exploratory Data Analysis

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label distribution
df['label'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('Label (1=Inhibitor)')
axes[0].set_ylabel('Count')

# pIC50 distribution
df['pIC50'].hist(bins=50, ax=axes[1])
axes[1].axvline(x=5, color='r', linestyle='--', label='10 µM threshold')
axes[1].set_title('pIC50 Distribution')
axes[1].set_xlabel('pIC50')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Visualize some example molecules
inhibitors = df[df['label'] == 1].head(8)
mols = [Chem.MolFromSmiles(s) for s in inhibitors['smiles']]
img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(300, 300))
img

## 4. Split Data

In [ ]:
from herg_predictor.data import get_split

df_train, df_val, df_test = get_split(
    df,
    strategy='scaffold',  # Use scaffold split for realistic evaluation
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
)

print(f"Train: {len(df_train)} ({df_train['label'].mean():.1%} positive)")
print(f"Val: {len(df_val)} ({df_val['label'].mean():.1%} positive)")
print(f"Test: {len(df_test)} ({df_test['label'].mean():.1%} positive)")

## 5. Compute Features

In [ ]:
from herg_predictor.features import featurize_fingerprints

# Compute Morgan fingerprints
X_train, train_valid = featurize_fingerprints(df_train['smiles'].tolist(), 'morgan')
X_val, val_valid = featurize_fingerprints(df_val['smiles'].tolist(), 'morgan')
X_test, test_valid = featurize_fingerprints(df_test['smiles'].tolist(), 'morgan')

y_train = df_train['label'].values[train_valid]
y_val = df_val['label'].values[val_valid]
y_test = df_test['label'].values[test_valid]

print(f"Feature shape: {X_train.shape}")

## 6. Train Models

In [ ]:
from herg_predictor.models import RandomForestModel, XGBoostModel, FeedForwardClassifier
from herg_predictor.evaluation import compute_classification_metrics

# Train Random Forest
rf_model = RandomForestModel(n_estimators=500)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict_proba(X_test)
rf_metrics = compute_classification_metrics(y_test, rf_pred)
print(f"Random Forest - AUROC: {rf_metrics['auroc']:.4f}, AUPRC: {rf_metrics['auprc']:.4f}")

In [ ]:
# Train XGBoost
xgb_model = XGBoostModel(n_estimators=500)
xgb_model.fit(X_train, y_train, X_val, y_val)

xgb_pred = xgb_model.predict_proba(X_test)
xgb_metrics = compute_classification_metrics(y_test, xgb_pred)
print(f"XGBoost - AUROC: {xgb_metrics['auroc']:.4f}, AUPRC: {xgb_metrics['auprc']:.4f}")

In [ ]:
# Train Feed-Forward Neural Network
ff_model = FeedForwardClassifier(
    input_dim=X_train.shape[1],
    hidden_dims=[512, 256, 128],
    dropout=0.3,
)
ff_model.fit(X_train, y_train, X_val, y_val, epochs=100, batch_size=64)

ff_pred = ff_model.predict_proba(X_test)
ff_metrics = compute_classification_metrics(y_test, ff_pred)
print(f"Feed-Forward NN - AUROC: {ff_metrics['auroc']:.4f}, AUPRC: {ff_metrics['auprc']:.4f}")

## 7. Compare Models

In [ ]:
from herg_predictor.evaluation.analysis import plot_roc_curve, plot_precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC curves
plot_roc_curve(y_test, rf_pred, ax=axes[0], label='Random Forest')
plot_roc_curve(y_test, xgb_pred, ax=axes[0], label='XGBoost')
plot_roc_curve(y_test, ff_pred, ax=axes[0], label='Feed-Forward NN')

# PR curves
plot_precision_recall_curve(y_test, rf_pred, ax=axes[1], label='Random Forest')
plot_precision_recall_curve(y_test, xgb_pred, ax=axes[1], label='XGBoost')
plot_precision_recall_curve(y_test, ff_pred, ax=axes[1], label='Feed-Forward NN')

plt.tight_layout()
plt.show()

## 8. Error Analysis

In [ ]:
from herg_predictor.evaluation.analysis import analyze_errors

# Analyze errors for best model
df_test_valid = df_test.iloc[test_valid].copy()
errors = analyze_errors(df_test_valid, y_test, xgb_pred)

print("Error Summary:")
print(errors['summary'])

print("\nTop False Positives (predicted inhibitor, actually not):")
print(errors['false_positives'][['smiles', 'y_pred_proba', 'activity_value']].head())